In [ ]:
StudentName_Medical_Imaging_Portfolio/
│
├── Task_1_Chest_XRay/
│   ├── data/                 # Specific X-ray images used for testing[cite: 86]
│   ├── output/               # Screenshots/saved images of enhanced results[cite: 86]
│   ├── xray_enhancement.ipynb # Python script[cite: 86]
│   └── README.md             # Brief explanation of processing pipeline[cite: 86]
│
├── Task_2_Cardiac_Fusion/
│   ├── data/                 # Specific CT and MRI matched pairs used[cite: 86]
│   ├── output/               # Saved fused heatmaps and comparison charts[cite: 86]
│   ├── modal_fusion.ipynb    # Python script[cite: 86]
│   └── README.md             # Fusion weighting logic and analysis[cite: 86]
│
├── Task_3_Echo_Analysis/
│   ├── data/                 # Short ultrasound mp4 snippet used[cite: 86]
│   ├── output/               # Screenshots of side-by-side video pipeline[cite: 86]
│   ├── realtime_echo.ipynb   # Python script[cite: 86]
│   └── README.md             # Instructions on how to run video loop[cite: 86]
│
├── requirements.txt          # Global library dependencies (e.g., opencv-python==4.8.0, numpy==1.26.0)[cite: 86, 87]
└── README.md                 # Main repository overview and setup instructions[cite: 86]
Task 1: Diagnostic Enhancement of Chest X-RaysThis pipeline maximizes structural visibility in severely underexposed single-channel X-ray matrices.   Pythonimport cv2
import numpy as np

# Load a sample grayscale image from the dataset[cite: 88]
image = cv2.imread('data/sample_xray.png', cv2.IMREAD_GRAYSCALE)

# Contrast Enhancement: Apply Histogram Equalization to forcefully redistribute pixel intensities[cite: 88]
equalized = cv2.equalizeHist(image)

# Color Mapping: Convert equalized grayscale into a false-color heatmap[cite: 88]
color_mapped = cv2.applyColorMap(equalized, cv2.COLORMAP_JET)

# Color Balance: Shift colors to neutralize artificial color casts[cite: 88]
# Example: Reducing the red channel intensity slightly
color_balanced = color_mapped.copy()
color_balanced[:, :, 2] = cv2.subtract(color_balanced[:, :, 2], 20)

# Color Filtering (Thresholding): Apply strict threshold to mask out everything except densest tissues[cite: 88]
_, dense_mask = cv2.threshold(image, 200, 255, cv2.THRESH_BINARY)

# Logarithmic Transformation: Expand dark background regions[cite: 88]
c_log = 255 / np.log(1 + np.max(image))
log_transformed = c_log * (np.log(1 + image.astype(np.float32)))
log_transformed = np.array(log_transformed, dtype=np.uint8)

# Power-Law (Gamma) Transformation: Fractional power-law (gamma < 1.0) to clear soft lung tissue[cite: 88, 89]
gamma = 0.6
c_gamma = 255 / (np.max(image) ** gamma)
gamma_transformed = c_gamma * (image.astype(np.float32) ** gamma)
gamma_transformed = np.array(gamma_transformed, dtype=np.uint8)
Task 2: Multi-Modal Cardiac Image FusionThis script mathematically fuses perfectly aligned CT and MRI slices of the same heart to combine structural boundaries with soft tissue variations.   Pythonimport cv2
import numpy as np

# Load one CT slice and its perfectly corresponding MRI slice[cite: 89]
ct_slice = cv2.imread('data/ct_slice.png', cv2.IMREAD_GRAYSCALE)
mri_slice = cv2.imread('data/mri_slice.png', cv2.IMREAD_GRAYSCALE)

# Apply equalization to both matrices independently to maximize dynamic ranges[cite: 89]
ct_eq = cv2.equalizeHist(ct_slice)
mri_eq = cv2.equalizeHist(mri_slice)

# Convert enhanced matrices into colored heatmaps[cite: 89]
ct_color = cv2.applyColorMap(ct_eq, cv2.COLORMAP_BONE)
mri_color = cv2.applyColorMap(mri_eq, cv2.COLORMAP_JET)

# Multi-Modal Weighted Fusion: Heavier weight to CT (edges), lighter weight to MRI (tissue)[cite: 89]
fused = cv2.addWeighted(ct_color, 0.7, mri_color, 0.3, 0)

# Pass final fused matrix through logarithmic and power-law functions to prevent data crushing[cite: 89]
fused_float = fused.astype(np.float32) / 255.0
fused_log = 255 * (np.log(1 + fused_float) / np.log(2))
fused_final = np.array(fused_log ** 0.8, dtype=np.uint8)

# Display standalone CT, standalone MRI, and fused output side-by-side[cite: 89, 90]
ct_display = cv2.cvtColor(ct_slice, cv2.COLOR_GRAY2BGR)
mri_display = cv2.cvtColor(mri_slice, cv2.COLOR_GRAY2BGR)
comparison = np.hstack((ct_display, mri_display, fused_final))
Task 3: Real-Time Echocardiogram Video AnalysisThis pipeline extracts dynamic ultrasound frames continuously, applying mathematical enhancements to live footage alongside the raw feed.   Pythonimport cv2
import numpy as np

# Initialize cv2.VideoCapture() to read from the provided ultrasound .mp4 file[cite: 90]
cap = cv2.VideoCapture('data/echo_sample.mp4')

# Start a while loop to extract frames continuously[cite: 90]
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Apply Histogram Equalization to combat murky ultrasound contrast[cite: 90]
    eq_frame = cv2.equalizeHist(gray_frame)

    # Convert frame to colored heatmap (COLORMAP_JET)[cite: 90]
    heat_frame = cv2.applyColorMap(eq_frame, cv2.COLORMAP_JET)

    # Apply a color balance adjustment[cite: 90]
    heat_frame[:, :, 0] = cv2.add(heat_frame[:, :, 0], 10)

    # Apply logarithmic transformation to reveal darkest regions[cite: 90]
    c_log = 255 / np.log(1 + np.max(heat_frame))
    log_frame = c_log * (np.log(1 + heat_frame.astype(np.float32)))

    # Apply power-law transformation to suppress blinding white backscatter noise[cite: 90]
    gamma = 1.5
    c_gamma = 255 / (255 ** gamma)
    final_frame = c_gamma * (log_frame ** gamma)
    final_frame = np.array(final_frame, dtype=np.uint8)

    # Concatenate frames to display raw, unedited stream side-by-side with enhanced pipeline[cite: 90]
    monitoring_array = np.hstack((frame, final_frame))
    cv2.imshow('Echocardiogram Analysis: Raw vs Enhanced', monitoring_array)

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

Photometric (or point) transformations operate directly on the individual pixel intensities of an image rather than altering their spatial coordinates. These mathematical operations change what the pixel looks like (e.g., brightness, contrast) rather than where it is located.Logarithmic Transformation ($s = c \log(1 + r)$): This mathematical curve expands the values of dark pixels while compressing the values of bright pixels. In medical imaging, it is used to mathematically expand dark background regions of a raw, underexposed X-ray, revealing faint outer edges of structures like the ribcage. It is also applied to echocardiograms to reveal the darkest regions of the heart chambers.   Power-Law / Gamma Transformation ($s = c r^\gamma$): This transformation dynamically adjusts midtones based on the fractional power value applied. Applying a fractional power-law transformation where $\gamma < 1.0$ brightens the image and reduces the harsh contrast of bones so that soft lung tissues become clearer. Conversely, manipulating this curve can suppress blinding white backscatter noise typical of ultrasound probes.   Piecewise-Linear Transformation (Thresholding): This involves applying different linear functions to distinct intensity ranges. A strict mathematical threshold acts as a piecewise step-function, pushing all pixels below a certain intensity to black and above it to white. This masks out everything except the densest tissues, such as bone or dense fluid, which reflect specific high-intensity values.   Histogram Equalization: This technique mathematically flattens the intensity distribution of an image to maximize contrast. It forcefully redistributes pixel intensities to combat murky contrast in ultrasounds and maximizes the dynamic range of CT and MRI matrices before blending.   Repository Architecture RequirementsYour submission must exactly follow this structural blueprint without dumping files into the main folder. GitHub file size limits require following the "100MB Rule": upload only the specific sample images or short video clips your scripts actually use, rather than the entire 1GB+ datasets. All Python scripts must use relative paths (e.g., cv2.imread('data/sample_xray.png')) rather than absolute local paths.